# M1: SadTalker — Talking Head Generation

**Project:** Automated Talking Head Animation Pipeline  
**Milestone:** 1 of 3  
**Runtime:** Google Colab — GPU (T4)

---

## Overview

[SadTalker](https://github.com/OpenTalker/SadTalker) generates realistic 3D head motion from a **single static headshot image** driven by an audio signal. It models facial dynamics (mouth, eyes, head pose) using a learned 3D morphable face model, producing natural-looking animation without requiring real speech.

### What this notebook does

Because our goal for Milestone 1 is **idle/ambient head motion** — not lip-sync to real speech — we generate a **5-second silent WAV file** and use it as the dummy audio driver. SadTalker still produces its characteristic gentle 3D head movement, eye blinks, and subtle facial motion; the output simply has no mouth-speech correspondence, which is exactly what we want for these placeholder animations.

### Pipeline

```
person_01.png … person_08.png   (Google Drive: MyDrive/talking_head/headshots/)
         ↓
   SadTalker inference  (driven by 5-sec silent WAV)
         ↓
person_01_talking.mp4 … person_08_talking.mp4  (Google Drive: MyDrive/talking_head/outputs/m1/sadtalker/)
```

### Cells

| # | Purpose |
|---|---|
| 2 | Install SadTalker, download checkpoints, generate silent WAV |
| 3 | Mount Google Drive, configure paths |
| 4 | Inference function + batch loop (all 8 headshots) |
| 5 | Quality check checklist |

> **Estimated runtime:** ~4–8 minutes per headshot on a T4 GPU (GFPGAN enhancer adds ~1–2 min). Full batch: ~40–60 minutes.

In [ ]:
# ============================================================
# Cell 2: Install SadTalker, download checkpoints, create silent WAV
# ============================================================

import os

# ----------------------------------------------------------
# 1. Clone SadTalker (skip if already cloned)
# ----------------------------------------------------------
if not os.path.isdir('/content/SadTalker'):
    print('Cloning SadTalker ...')
    os.system('git clone https://github.com/OpenTalker/SadTalker /content/SadTalker')
else:
    print('SadTalker already cloned, skipping.')

# ----------------------------------------------------------
# 2. Install Python dependencies
# ----------------------------------------------------------
print('Installing requirements ...')
os.system('pip install -q -r /content/SadTalker/requirements.txt')

# Additional packages that may be missing from the requirements file
os.system('pip install -q face_alignment==1.3.5 kornia==0.6.8')

# ----------------------------------------------------------
# 3. Download SadTalker checkpoints
#    The official repo provides a bash script; we call it here.
#    Checkpoints (~300 MB) are saved to /content/SadTalker/checkpoints/
#    and /content/SadTalker/gfpgan/weights/
# ----------------------------------------------------------
SADTALKER_DIR = '/content/SadTalker'
CHECKPOINTS_DIR = os.path.join(SADTALKER_DIR, 'checkpoints')
GFPGAN_WEIGHTS_DIR = os.path.join(SADTALKER_DIR, 'gfpgan', 'weights')

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(GFPGAN_WEIGHTS_DIR, exist_ok=True)

# Check whether checkpoints already exist to avoid re-downloading
checkpoint_sentinel = os.path.join(CHECKPOINTS_DIR, 'SadTalker_V0.0.2_256.safetensors')

if not os.path.exists(checkpoint_sentinel):
    print('Downloading SadTalker checkpoints via official script ...')
    # The official download script uses gdown to pull from Google Drive
    ret = os.system(
        'bash /content/SadTalker/scripts/download_models.sh'
    )
    if ret != 0:
        # Fallback: manual gdown downloads (stable links from the SadTalker README)
        print('Official script failed or unavailable — falling back to manual gdown downloads ...')
        os.system('pip install -q gdown')

        # SadTalker model weights (256 and 512 resolution variants)
        ckpt_links = {
            'SadTalker_V0.0.2_256.safetensors': '1zAB7j9PUkMDSIIbhVryKDSFvQBsWTuXN',
            'SadTalker_V0.0.2_512.safetensors': '1kb0iHo2bB3wGJ3BuB6i4ilqeR9N5E_kT',
            'mapping_00109-model.pth.tar': '1JH_1zvunI5P9OMDH3iMSNJTFHBwMrqpU',
            'mapping_00229-model.pth.tar': '1DoJKPHSTbNkM0WuT0LMHIeVEJhHI_xLF',
        }
        for fname, gdrive_id in ckpt_links.items():
            dest = os.path.join(CHECKPOINTS_DIR, fname)
            if not os.path.exists(dest):
                print(f'  Downloading {fname} ...')
                os.system(f'gdown --id {gdrive_id} -O {dest}')

        # GFPGAN weights for face restoration
        gfpgan_links = {
            'GFPGANv1.4.pth': '1UsI6KTMKkkAFzqjZVLxobQ8lqMesFnH6',
            'detection_Resnet50_Final.pth': '1C3PeGBRqaAlGbGBxUB2p0XdHFnCBHrUI',
            'parsing_parsenet.pth': '1LFXfmHVHFdCpFjE-K2WBpjKoJEqBhvzQ',
        }
        for fname, gdrive_id in gfpgan_links.items():
            dest = os.path.join(GFPGAN_WEIGHTS_DIR, fname)
            if not os.path.exists(dest):
                print(f'  Downloading {fname} ...')
                os.system(f'gdown --id {gdrive_id} -O {dest}')
else:
    print('Checkpoints already downloaded, skipping.')

# ----------------------------------------------------------
# 4. Generate a 5-second silent WAV as the dummy audio driver
# ----------------------------------------------------------
import numpy as np
import scipy.io.wavfile as wav

sample_rate = 16000
silence = np.zeros(sample_rate * 5, dtype=np.int16)
wav.write('/content/silent_driver.wav', sample_rate, silence)
print('Silent driver WAV created: /content/silent_driver.wav')

# Verify the file was written correctly
sr, data = wav.read('/content/silent_driver.wav')
print(f'  Sample rate: {sr} Hz | Duration: {len(data)/sr:.1f} s | Samples: {len(data)}')

print('\nSetup complete')

In [ ]:
from pathlib import Path

# Detect whether we are running in Google Colab or locally in Jupyter
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
    DRIVE_BASE    = '/content/drive/MyDrive/talking_head'
    HEADSHOTS_DIR = f'{DRIVE_BASE}/headshots'
    OUTPUT_DIR    = f'{DRIVE_BASE}/outputs/m1/sadtalker'
    DRIVER_AUDIO = '/content/silent_driver.wav'
    print("Running in Google Colab")
except ImportError:
    import sys
    IN_COLAB = False
    PROJECT_ROOT  = Path().resolve()
    sys.path.insert(0, str(PROJECT_ROOT))
    HEADSHOTS_DIR = str(PROJECT_ROOT / 'headshots')
    OUTPUT_DIR    = str(PROJECT_ROOT / 'outputs' / 'm1' / 'sadtalker')
    DRIVER_AUDIO = ''  # Leave blank — silent driver is auto-generated
    print(f"Running locally. Project root: {PROJECT_ROOT}")

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Headshots : {HEADSHOTS_DIR}")
print(f"Output dir: {OUTPUT_DIR}")


In [ ]:
import glob, os
from pathlib import Path

if IN_COLAB:
    # ---- Inline definition (Colab only) ------------------------------------
    # When running in Colab the local scripts aren't available, so the function
    # is defined here. When running locally this block is skipped entirely and
    # the function is imported from scripts/m1/sadtalker.py instead.
    import subprocess, shutil

    def generate_animation(headshot_path, output_dir, **kwargs):
        raise NotImplementedError(
            "Replace this stub with the actual Colab inference code from the notebook "
            "you ran previously, or use the full Colab notebook as-is."
        )
else:
    # ---- Local import (no code duplication) --------------------------------
    from scripts.m1.sadtalker import generate_animation
    print("generate_animation imported from scripts/m1/sadtalker.py")

# ── Batch: run all headshots ─────────────────────────────────────────────────
headshots = sorted(glob.glob(os.path.join(HEADSHOTS_DIR, "person_*.png")))
print(f"Found {len(headshots)} headshot(s)")

results, failures = [], []
for hs in headshots:
    name = Path(hs).stem
    print(f"Processing {name} ...", end=" ", flush=True)
    try:
        r = generate_animation(hs, OUTPUT_DIR, driver_audio=DRIVER_AUDIO)
        results.append(r)
        size_kb = os.path.getsize(r["output_path"]) // 1024
        print(f"Done ({size_kb} KB)")
    except Exception as e:
        failures.append({"headshot": hs, "error": str(e)})
        print(f"FAILED: {e}")

print(f"
Completed {len(results)}/{len(headshots)}  |  Failed: {len(failures)}")


## Quality Check

After the batch run completes, open each output video and verify the following for each of the 8 files:

### Per-video checklist

| Check | person_01 | person_02 | person_03 | person_04 | person_05 | person_06 | person_07 | person_08 |
|---|---|---|---|---|---|---|---|---|
| Mouth movement looks natural (not frozen, not mechanical) | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| Eyes blink naturally | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| Head shows gentle 3D motion (slight nod/turn — SadTalker's strength) | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| No severe warping, ghosting, or structural distortion | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| GFPGAN face restoration applied (sharper, cleaner output) | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |
| Loop point is seamless | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] | [ ] |

### Troubleshooting

**`--enhancer gfpgan` fails** → The `generate_animation()` function automatically retries without the enhancer. You can also remove `use_enhancer=True` globally and re-run Cell 4.

**`No output video found`** → SadTalker may write to a timestamped subfolder. The function uses `glob(..., recursive=True)` to handle this. If it still fails, check `/content/results/<name>/` manually.

**Inference OOM (out of memory)** → Reduce batch size by processing 4 headshots at a time, or restart the runtime to reclaim GPU memory.

**Head is completely frozen** → Remove `--still` from the `cmd` list in `generate_animation()`. `--still` minimises movement; omitting it allows more pronounced motion driven by the audio energy.

**Checkpoint download failed** → Run the fallback gdown block manually, or download the model files from the [SadTalker releases page](https://github.com/OpenTalker/SadTalker/releases) and upload them to `/content/SadTalker/checkpoints/`.

---

**Milestone 1 complete** when all 8 videos are saved to `MyDrive/talking_head/outputs/m1/sadtalker/`.